In [ ]:
!pip install praw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 4.4 MB/s eta 0:00:00


In [ ]:
!pip install python-dotenv

In [ ]:
import praw
from praw.models import MoreComments
from datetime import timedelta,timezone, datetime
import time
import pandas as pd
import json
from dotenv import load_dotenv
import os
from spacy import displacy
import re


In [ ]:
load_dotenv()

# Reddit config
reddit = praw.Reddit(
    client_id = os.getenv("CLIENT_ID"),
    client_secret = os.getenv("CLIENT_SECRET"),
    user_agent=os.getenv("AGENT"),
    ratelimit_seconds=300
)



In [ ]:
def per_subreddit(sub, keywords):
    data_posts = []
    data_comments = []

    subreddit = reddit.subreddit(sub)
    limits = 1000

    # data de extração (agora)
    extraction_date = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S %Z')
    for keyword in keywords:
      print(f"🔎 Buscando '{keyword}' em r/{sub}")

      for post in subreddit.search(keyword, limit=limits, sort="new", time_filter="year"):

        submission_date = datetime.fromtimestamp(post.created_utc, tz=timezone.utc)
        format_date_post = submission_date.strftime('%Y-%m-%d %H:%M:%S %Z')

        # 👇 NOVO BLOCO (AQUI É O PONTO IMPORTANTE)
        text = ((post.title or "") + " " + (post.selftext or "")).lower()

        matched_keywords = [
            k for k in keywords
            if re.search(rf"\b{k}\b", text)]

        # 👇 SEU APPEND MODIFICADO
        data_posts.append({
            "ID": post.id,
            "Date": format_date_post,
            "Extraction_date": extraction_date,
            "Subreddit": str(post.subreddit),
            "URL": post.url,
            "Title": post.title,
            "Text": post.selftext,
            "Author": str(post.author),
            "Score": post.score,
            "Num_comment": post.num_comments,
            "Keyword": keyword,  # termo buscado
            "Matched_keywords": ", ".join(matched_keywords)  # 👈 NOVO
        })


        time.sleep(1)

        # garante que todos os comentários sejam carregados
        post.comments.replace_more(limit=0)

        for comment in post.comments:
            if isinstance(comment, MoreComments):
                continue

            dt_utc_comment = datetime.fromtimestamp(comment.created_utc, tz=timezone.utc)
            format_date_comment = dt_utc_comment.strftime('%Y-%m-%d %H:%M:%S %Z')

            data_comments.append({
                "Comment_id": comment.id,
                "Date_comment": format_date_comment,
                "Extraction_date": extraction_date,
                "Subreddit": sub,
                "ID_post": post.id,
                "URL": post.url,
                "Comment": comment.body,
                "Author": str(comment.author),
                "Is_submitter": comment.is_submitter,
                "Score": comment.score
            })

    return data_posts, data_comments

In [ ]:
def main():
    subreddits = ["DadosBrasil", "Dados","datasciencebr","ProgramadoresBrasil", "brdev","devBR"]  # 👈 lista de subs
    keywords = ["copilot","github copilot", "cursor ai","cursor ide" , "codewhisperer","amazon codewhisperer", "codeium","tabnine","chatGPT","claude","gemini","google gemini",
                "deepseek","diffblue cover","codiumAI","snyk code","replit ghostwriter","v0 by vercel","v0 vercel","lovable ai","notion AI","harness","pulumi AI","bard","chat-gpt","gpt", "gpt-4"]

    all_posts = []
    all_comments = []

    for sub in subreddits:
        print(f"Coletando: {sub}")
        time.sleep(2)  #
        posts, comments = per_subreddit(sub,keywords)

        all_posts.extend(posts)
        all_comments.extend(comments)

    df_post = pd.DataFrame(all_posts)
    df_comment = pd.DataFrame(all_comments)

    df_post.to_csv("posts.csv", index=False)
    df_comment.to_csv("comments.csv", index=False)

In [ ]:
main()

Coletando: DadosBrasil


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



🔎 Buscando 'copilot' em r/DadosBrasil


Forbidden: received 403 HTTP response